In [1]:
# Install required packages (run this cell first)
!pip install -q sentence-transformers rank-bm25 torch numpy scikit-learn google-generativeai groq

  You can safely remove it manually.


In [9]:

import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model_gemini = genai.GenerativeModel('gemini-2.5-flash')

print("Setup complete and API configured.")

Setup complete and API configured.


In [10]:
corpus = [
    # Neural Network Training (3+ related docs)
    "Neural networks are trained using backpropagation, where gradients are computed through the chain rule and weights are updated via gradient descent.",
    "Training deep neural networks requires optimization techniques like Adam, which adapts learning rates for each parameter using momentum and RMSProp.",
    "Overfitting in neural network training is prevented using techniques like dropout, weight decay, early stopping, and data augmentation.",
    "Batch normalization stabilizes neural network training by normalizing layer inputs, reducing internal covariate shift during gradient updates.",
    
    # Transformers & Attention
    "The attention mechanism in transformers computes weighted sums of values based on query-key similarity, enabling parallel sequence processing.",
    "Self-attention in transformers allows each position to attend to all positions, capturing long-range dependencies without recurrence.",
    "Multi-head attention in transformers runs multiple attention mechanisms in parallel, allowing the model to focus on different representation subspaces.",
    
    # Optimization
    "Stochastic gradient descent (SGD) with momentum accelerates training by accumulating velocity from past gradients to overcome local minima.",
    "AdamW optimizer decouples weight decay from the adaptive learning rate in Adam, improving regularization in transformer training.",
    
    # Technical jargon doc (BM25 should excel)
    "ReLU activation function (f(x) = max(0,x)) and AdamW optimizer with cosine annealing LR schedule are crucial for training Vision Transformers (ViT).",
    
    # Other ML topics
    "Reinforcement learning agents learn policies by maximizing expected cumulative reward through value function approximation or policy gradients.",
    "Hidden Markov Models (HMMs) model sequential data with hidden states using Viterbi algorithm for decoding most likely state sequences.",
    "Principal Component Analysis (PCA) reduces dimensionality by projecting data onto directions of maximum variance."
]

print(f"Created corpus with {len(corpus)} documents:")
for i, doc in enumerate(corpus[:3]):  # Show first 3
    print(f"{i}: {doc[:100]}...")
print("...and more.")

doc_ids = list(range(len(corpus)))

Created corpus with 13 documents:
0: Neural networks are trained using backpropagation, where gradients are computed through the chain ru...
1: Training deep neural networks requires optimization techniques like Adam, which adapts learning rate...
2: Overfitting in neural network training is prevented using techniques like dropout, weight decay, ear...
...and more.


In [11]:
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        # Initialize BM25 (Keyword search)
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)
        # Initialize SBERT (Semantic search)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus_embeddings = self.embedder.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # 1. BM25 Ranking
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]
        
        # 2. SBERT Ranking
        query_embedding = self.embedder.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0]
        sbert_ranks = torch.argsort(cos_scores, descending=True).cpu().numpy()

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}
        for rank, idx in enumerate(bm25_ranks):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (self.k + rank)
        for rank, idx in enumerate(sbert_ranks):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (self.k + rank)

        # Sort by RRF score
        sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
        
        results = []
        for idx in sorted_indices[:top_k]:
            results.append({
                "doc_id": idx,
                "rrf_score": rrf_scores[idx],
                "bm25_rank": int(np.where(bm25_ranks == idx)[0][0]),
                "sbert_rank": int(np.where(sbert_ranks == idx)[0][0]),
                "text": self.corpus[idx]
            })
        return results

# Initialize the retriever
retriever = HybridRetriever(corpus)

In [12]:
from sentence_transformers import CrossEncoder

# Part 3: Cross-Encoder Re-Ranker
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query: str, candidates: list[dict], top_k: int = 3):
    texts = [c['text'] for c in candidates]
    pairs = [[query, text] for text in texts]
    scores = cross_encoder.predict(pairs)
    
    for i, score in enumerate(scores):
        candidates[i]['cross_score'] = score
        
    # Higher score = more relevant
    reranked = sorted(candidates, key=lambda x: x['cross_score'], reverse=True)
    return reranked[:top_k]

# Part 4: Query Expansion (Option A - HyDE)
def generate_hyde_query(query: str):
    # Temperature 0.0 for factual/deterministic results
    prompt = f"Write a one-sentence technical answer to: {query}"
    response = model_gemini.generate_content(prompt, generation_config={"temperature": 0.0})
    return response.text

In [13]:
def advanced_rag(user_query: str) -> str:
    # 1. Query Expansion
    expanded_query = generate_hyde_query(user_query)
    
    # 2. Hybrid Retrieval (Retrieving slightly more to allow re-ranking room)
    candidates = retriever.retrieve(expanded_query, top_k=6)
    
    # 3. Re-Ranking using original query
    top_docs = rerank(user_query, candidates, top_k=3)
    
    # 4. LLM Generation
    context = "\n".join([f"- {d['text']}" for d in top_docs])
    final_prompt = f"""You are a university assistant. Use the context below to answer the student's question.
    
Context:
{context}

Question: {user_query}
Answer:"""
    
    response = model_gemini.generate_content(final_prompt)
    return response.text

# Naive RAG for comparison (Dense only, no expansion, no rerank)
def naive_rag_top_doc(query: str):
    query_embedding = retriever.embedder.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, retriever.corpus_embeddings)[0]
    best_idx = torch.argmax(cos_scores).item()
    return corpus[best_idx]

In [14]:


test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what are the main vector databases?"
]

comparison_data = []

for q in test_queries:
    naive_top = naive_rag_top_doc(q)
    
    # Advanced logic to get just the top doc after re-ranking
    exp_q = generate_hyde_query(q)
    cand = retriever.retrieve(exp_q, top_k=5)
    adv_top = rerank(q, cand, top_k=1)[0]['text']
    
    comparison_data.append({
        "Query": q,
        "Naïve RAG Top Doc": naive_top,
        "Advanced RAG Top Doc": adv_top,
        "Are they different?": "Yes" if naive_top != adv_top else "No"
    })

df = pd.DataFrame(comparison_data)
df

,Query,Naïve RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Self-attention in transformers allows each pos...,The attention mechanism in transformers comput...,Yes
1,optimization techniques for training,Training deep neural networks requires optimiz...,Training deep neural networks requires optimiz...,No
2,what are the main vector databases?,Principal Component Analysis (PCA) reduces dim...,The attention mechanism in transformers comput...,Yes


In [16]:
def weighted_rrf_retrieval(query: str, alpha: float = 0.5, k: int = 60, top_k: int = 3):
    """
    Implements Weighted Reciprocal Rank Fusion:
    RRF = alpha * (1 / (k + r_bm25)) + (1 - alpha) * (1 / (k + r_sbert))
    """
    # 1. Get Base Ranks
    tokenized_query = query.lower().split()
    bm25_scores = retriever.bm25.get_scores(tokenized_query)
    bm25_ranks = np.argsort(bm25_scores)[::-1]
    
    query_embedding = retriever.embedder.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, retriever.corpus_embeddings)[0]
    sbert_ranks = torch.argsort(cos_scores, descending=True).cpu().numpy()

    # 2. Apply Weighted Fusion
    weighted_scores = {}
    for rank, idx in enumerate(bm25_ranks):
        weighted_scores[idx] = weighted_scores.get(idx, 0) + alpha * (1 / (k + rank))
    for rank, idx in enumerate(sbert_ranks):
        weighted_scores[idx] = weighted_scores.get(idx, 0) + (1 - alpha) * (1 / (k + rank))

    sorted_indices = sorted(weighted_scores.keys(), key=lambda x: weighted_scores[x], reverse=True)
    return [(retriever.corpus[i], weighted_scores[i]) for i in sorted_indices[:top_k]]

# --- Bonus Experiment: Keyword vs Semantic Focus ---
test_query = "Adam optimizer deep learning" # Jargon heavy

print(f"Query: '{test_query}'\n")

# Scenario 1: Keyword Focus (High Alpha)
res_high_alpha = weighted_rrf_retrieval(test_query, alpha=0.8)
print(f"Alpha 0.8 (Keyword Heavy) Top Doc: {res_high_alpha[0][0]}")

# Scenario 2: Balanced (Standard RRF)
res_mid_alpha = weighted_rrf_retrieval(test_query, alpha=0.5)
print(f"Alpha 0.5 (Balanced) Top Doc: {res_mid_alpha[0][0]}")

# Scenario 3: Semantic Focus (Low Alpha)
res_low_alpha = weighted_rrf_retrieval(test_query, alpha=0.2)
print(f"Alpha 0.2 (Semantic Heavy) Top Doc: {res_low_alpha[0][0]}")

Query: 'Adam optimizer deep learning'

Alpha 0.8 (Keyword Heavy) Top Doc: Training deep neural networks requires optimization techniques like Adam, which adapts learning rates for each parameter using momentum and RMSProp.
Alpha 0.5 (Balanced) Top Doc: Training deep neural networks requires optimization techniques like Adam, which adapts learning rates for each parameter using momentum and RMSProp.
Alpha 0.2 (Semantic Heavy) Top Doc: Training deep neural networks requires optimization techniques like Adam, which adapts learning rates for each parameter using momentum and RMSProp.


In [18]:
import pandas as pd
import time

# --- Part 6: Comparison Experiment ---

# 1. Define the 3 test queries as per requirements
test_queries = [
    "how do transformers encode meaning?",           # Required Query 1
    "optimization techniques for training",         # Required Query 2
    "what are the main vector databases?"           # User Query 3 (Your own)
]

comparison_results = []

print("Running Comparison Experiment (Naïve vs. Advanced RAG)...")

for q in test_queries:
    try:
        # --- NAÏVE RAG ---
        # Dense-only retrieval (SBERT cosine, no expansion, no re-ranking)
        naive_top_doc = naive_rag_top_doc(q)
        
        # --- ADVANCED RAG ---
        # Full pipeline: Query Expansion (HyDE) → Hybrid Retrieval (Weighted RRF) → Re-Ranking
        
        # A. Query Expansion
        hyde_q = generate_hyde_query(q)
        
        # B. Hybrid Retrieval (Using alpha=0.5 for balanced RRF)
        # Note: we retrieve more (top_k=6) to give the re-ranker enough candidates
        candidates = retriever.retrieve(hyde_q, top_k=6)
        
        # C. Re-Ranking (using original user query)
        reranked_results = rerank(q, candidates, top_k=1)
        advanced_top_doc = reranked_results[0]['text']
        
        # Record results (truncating text for clean table display)
        comparison_results.append({
            "Query": q,
            "Naïve RAG Top Doc": (naive_top_doc[:75] + '...') if len(naive_top_doc) > 75 else naive_top_doc,
            "Advanced RAG Top Doc": (advanced_top_doc[:75] + '...') if len(advanced_top_doc) > 75 else advanced_top_doc,
            "Are they different?": "Yes" if naive_top_doc != advanced_top_doc else "No"
        })
        
        # Rate limiting safety for Gemini API
        time.sleep(1.5)
        
    except Exception as e:
        print(f"Error processing query '{q}': {e}")

# 2. Create DataFrame
df_comparison = pd.DataFrame(comparison_results)

# 3. Output the Final Table in Markdown format for the assignment
print("\n" + "="*30)
print("FINAL ASSIGNMENT COMPARISON TABLE")
print("="*30 + "\n")
print(df_comparison.to_markdown(index=False))

# Also display the styled dataframe for the notebook UI
df_comparison

Running Comparison Experiment (Naïve vs. Advanced RAG)...

FINAL ASSIGNMENT COMPARISON TABLE

| Query                                | Naïve RAG Top Doc                                                              | Advanced RAG Top Doc                                                           | Are they different?   |
|:-------------------------------------|:-------------------------------------------------------------------------------|:-------------------------------------------------------------------------------|:----------------------|
| how do transformers encode meaning?  | Self-attention in transformers allows each position to attend to all positi... | The attention mechanism in transformers computes weighted sums of values ba... | Yes                   |
| optimization techniques for training | Training deep neural networks requires optimization techniques like Adam, w... | Training deep neural networks requires optimization techniques like Adam, w... | No                    

,Query,Naïve RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Self-attention in transformers allows each pos...,The attention mechanism in transformers comput...,Yes
1,optimization techniques for training,Training deep neural networks requires optimiz...,Training deep neural networks requires optimiz...,No
2,what are the main vector databases?,Principal Component Analysis (PCA) reduces dim...,Hidden Markov Models (HMMs) model sequential d...,Yes
